# 05a — Filtering with Random Scores

Sanity check for the `filter_new_token` mechanism using uniformly
random scores. With random scores the new token has a
`n_kept / num_scored` probability of being kept at each step, which
converges to `1 - compression_ratio`. This isolates the filtering
logic from any scoring bias (e.g., KeyDiff's anchor effect).

Key findings from this notebook:
- Using `num_scored` (actual cache size + 1) for the threshold gives
  an unbiased ~50% keep rate; using `total_tokens_seen` inflates it
- `round()` instead of `int()` reduces truncation bias at small cache sizes
- Majority vote with an even number of heads (e.g., 4) inflates the
  keep rate to ~69% because ties (2/4) always resolve to KEEP
- A single head gives the expected ~50% keep rate

## Imports and Setup

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)
torch.set_grad_enabled(False)

assert torch.cuda.is_available(), "CUDA GPU required"
print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configuration

In [ ]:
NUM_KV_HEADS = 1
HEAD_SIZE = 128
PREFILL_LEN = 0
NUM_DECODE_STEPS = 16384
COMPRESSION_RATIO = 0.5
DTYPE = torch.float16
DEVICE = "cuda"

## Filtering Function

In [ ]:
def keydiff_score(keys):
    """Score keys using KeyDiff's key-similarity metric.

    keys: [seq_len, num_kv_heads, head_dim]
    Returns: [num_kv_heads, seq_len]. Higher scores = more important.
    """
    keys_by_head = keys.permute(1, 0, 2)
    normalized = F.normalize(keys_by_head, p=2, dim=-1)
    anchor = normalized.mean(dim=1, keepdim=True)
    scores = -F.cosine_similarity(keys_by_head, anchor, dim=-1)
    return scores


def filter_new_token(scores, num_scored, compression_ratio):
    """Decide per head whether the newest token (last position) scores
    above the top-k threshold, where k = num_scored * (1 - ratio).
    """
    n_kept = max(1, round(num_scored * (1 - compression_ratio)))
    threshold = scores.topk(n_kept, dim=-1, sorted=True).values[:, -1]
    return scores[:, -1] >= threshold


print("Scoring and filtering functions defined")

## Experiment — Random Scores

At each decode step, generate random scores for the current cache
plus the new token, and apply the filtering threshold.

In [ ]:
cache_len = PREFILL_LEN
decisions = []

for step in range(NUM_DECODE_STEPS):
    num_scored = cache_len + 1
    scores = torch.rand(NUM_KV_HEADS, num_scored, device=DEVICE)

    keep_per_head = filter_new_token(scores, num_scored, COMPRESSION_RATIO)
    keep = keep_per_head.float().mean() >= 0.5

    decisions.append(keep.item())

    if keep:
        cache_len += 1

kept_count = sum(decisions)
skipped_count = NUM_DECODE_STEPS - kept_count

print(f"Results (random scores):")
print(f"  Kept:    {kept_count}/{NUM_DECODE_STEPS} decode tokens")
print(f"  Skipped: {skipped_count}/{NUM_DECODE_STEPS} decode tokens")
print(f"  Final cache size: {cache_len} tokens")
print(f"  Keep rate: {kept_count / NUM_DECODE_STEPS:.1%}")

## Experiment — KeyDiff Scores

Same filtering loop but scoring random keys with `keydiff_score`.
The anchor bias should inflate the keep rate above 50%, even though
the filtering logic itself is unbiased.

In [ ]:
all_keys = torch.randn(
    PREFILL_LEN + NUM_DECODE_STEPS, NUM_KV_HEADS, HEAD_SIZE,
    dtype=DTYPE, device=DEVICE,
)

cached_keys = all_keys[:PREFILL_LEN].clone() if PREFILL_LEN > 0 else \
    torch.empty(0, NUM_KV_HEADS, HEAD_SIZE, dtype=DTYPE, device=DEVICE)
decisions_kd = []

for step in range(NUM_DECODE_STEPS):
    new_key = all_keys[PREFILL_LEN + step]

    keys_with_new = torch.cat(
        [cached_keys, new_key.unsqueeze(0)], dim=0,
    )

    scores = keydiff_score(keys_with_new)
    num_scored = keys_with_new.shape[0]
    keep_per_head = filter_new_token(scores, num_scored, COMPRESSION_RATIO)
    keep = keep_per_head.float().mean() >= 0.5

    decisions_kd.append(keep.item())

    if keep:
        cached_keys = keys_with_new

kept_count = sum(decisions_kd)
skipped_count = NUM_DECODE_STEPS - kept_count

print(f"Results (KeyDiff scores):")
print(f"  Kept:    {kept_count}/{NUM_DECODE_STEPS} decode tokens")
print(f"  Skipped: {skipped_count}/{NUM_DECODE_STEPS} decode tokens")
print(f"  Final cache size: {cached_keys.shape[0]} tokens")
print(f"  Keep rate: {kept_count / NUM_DECODE_STEPS:.1%}")

## Experiment — KeyDiff Scores (total_tokens_seen)

Same as above but using `total_tokens_seen` for the threshold instead
of `num_scored`. With real scoring the mismatch matters: `n_kept` grows
with every step while scores are computed over a smaller, self-selected
cache.

In [ ]:
cached_keys_tts = all_keys[:PREFILL_LEN].clone() if PREFILL_LEN > 0 else \
    torch.empty(0, NUM_KV_HEADS, HEAD_SIZE, dtype=DTYPE, device=DEVICE)
decisions_kd_tts = []

for step in range(NUM_DECODE_STEPS):
    total_tokens_seen = PREFILL_LEN + step + 1
    new_key = all_keys[PREFILL_LEN + step]

    keys_with_new = torch.cat(
        [cached_keys_tts, new_key.unsqueeze(0)], dim=0,
    )

    scores = keydiff_score(keys_with_new)
    keep_per_head = filter_new_token(scores, total_tokens_seen, COMPRESSION_RATIO)
    keep = keep_per_head.float().mean() >= 0.5

    decisions_kd_tts.append(keep.item())

    if keep:
        cached_keys_tts = keys_with_new

kept_count = sum(decisions_kd_tts)
skipped_count = NUM_DECODE_STEPS - kept_count

print(f"Results (KeyDiff scores, total_tokens_seen):")
print(f"  Kept:    {kept_count}/{NUM_DECODE_STEPS} decode tokens")
print(f"  Skipped: {skipped_count}/{NUM_DECODE_STEPS} decode tokens")
print(f"  Final cache size: {cached_keys_tts.shape[0]} tokens")
print(f"  Keep rate: {kept_count / NUM_DECODE_STEPS:.1%}")

## Experiment — KeyDiff Scores (total_tokens_seen + int)

Same as above but using `int()` instead of `round()` for the threshold,
matching kvpress's original implementation exactly.

In [ ]:
def filter_new_token_int(scores, num_scored, compression_ratio):
    n_kept = max(1, int(num_scored * (1 - compression_ratio)))
    threshold = scores.topk(n_kept, dim=-1, sorted=True).values[:, -1]
    return scores[:, -1] >= threshold


cached_keys_int = all_keys[:PREFILL_LEN].clone() if PREFILL_LEN > 0 else \
    torch.empty(0, NUM_KV_HEADS, HEAD_SIZE, dtype=DTYPE, device=DEVICE)
decisions_kd_int = []

for step in range(NUM_DECODE_STEPS):
    total_tokens_seen = PREFILL_LEN + step + 1
    new_key = all_keys[PREFILL_LEN + step]

    keys_with_new = torch.cat(
        [cached_keys_int, new_key.unsqueeze(0)], dim=0,
    )

    scores = keydiff_score(keys_with_new)
    keep_per_head = filter_new_token_int(scores, total_tokens_seen, COMPRESSION_RATIO)
    keep = keep_per_head.float().mean() >= 0.5

    decisions_kd_int.append(keep.item())

    if keep:
        cached_keys_int = keys_with_new

kept_count = sum(decisions_kd_int)
skipped_count = NUM_DECODE_STEPS - kept_count

print(f"Results (KeyDiff scores, total_tokens_seen, int):")
print(f"  Kept:    {kept_count}/{NUM_DECODE_STEPS} decode tokens")
print(f"  Skipped: {skipped_count}/{NUM_DECODE_STEPS} decode tokens")
print(f"  Final cache size: {cached_keys_int.shape[0]} tokens")
print(f"  Keep rate: {kept_count / NUM_DECODE_STEPS:.1%}")